In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD


In [2]:
#loading the Review dataset
df=pd.read_csv(r"C:\Users\HP\Desktop\Bookify\Database\Cleaned_Datasets\preprocessed_review.csv")

In [3]:
df.head()

,review_id,book_id,user_id,review_text,ratings,review_date,source,author,book_name,cleaned_tokens
0,1,B0033UV8HI,A3HHXRELK8BHQG,Jace Rankin may be short but hes nothing to me...,3,2010-09-02,Amazon,Unknown,Unknown,jace rankin may short hes nothing mess man hau...
1,2,B002HJV4DE,A2RGNZ0TRF578I,Great short read I didnt want to put it down s...,5,2013-10-08,Amazon,Unknown,Unknown,great short read didnt want put read one sitti...
2,3,B002ZG96I4,A3S0H2HV6U1I7F,Ill start by saying this is the first of four ...,3,2014-04-11,Amazon,Unknown,Unknown,ill start saying first four books wasnt expect...
3,4,B002QHWOEU,AC4OQW3GZ919J,Aggie is Angela Lansbury who carries pocketboo...,3,2014-07-05,Amazon,Unknown,Unknown,aggie angela lansbury carries pocketbooks inst...
4,5,B001A06VJ8,A3C9V987IQHOQD,I did not expect this type of book to be in li...,4,2012-12-31,Amazon,Unknown,Unknown,expect type library pleased find price right


### 1. Prepare the Interaction Matrix

In [4]:
# Create the user-item interaction matrix
interaction_matrix = df.pivot_table(index='user_id', columns='book_id', values='ratings', fill_value=0)

print(f"Interaction matrix shape: {interaction_matrix.shape}")
interaction_matrix.head()

Interaction matrix shape: (8423, 2206)


book_id,006332752X,038554734X,059323006X,067144901X,078525224X,110199715X,1250787653,1250866448,1338233580,1338635174,...,B004NSV5DG,B004NSV8JC,B004NSVQ6M,B004O0U7QY,B004O0UA1Q,B09CGGV8DX,B09KN2QCML,B0B6XFT4RH,B0BW31X61X,unknown
user_id,,,,,,,,,,,,,,,,,,,,,
A H Kobayashi,0,0,0,0,0,0,3,0,0,0,...,0,0,0,0,0,0,0,0,0,0.0
A K P,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.0
A Reviewer,0,0,0,0,0,4,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.0
A Slater,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.0
A0089401235VSN3Z6F3HK,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.0


### 2. User-User Collaborative Filtering

In [5]:
# Create a User-Item matrix
interaction_matrix_filled = interaction_matrix.fillna(0)  # Fill missing values with 0

# Prepare the feature matrix for KNNClassifier
X = interaction_matrix_filled.values

# Train a NearestNeighbors model
knn_model = NearestNeighbors(metric='cosine', n_neighbors=6)  
knn_model.fit(X)  


NearestNeighbors(metric='cosine', n_neighbors=6)

In [6]:
# Function to get User-User Recommendations
def user_user_knn_recommendations(user_id, top_n=5, k=5):
    user_idx = interaction_matrix_filled.index.get_loc(user_id)

    # Predict similar users using the classifier
    distances, indices = knn_model.kneighbors([X[user_idx]], n_neighbors=k+1)

    similar_users = interaction_matrix_filled.index[indices.flatten()[1:]]
    
    #  Collect books liked (rated > 3) by similar users
    recommended_books = []
    for sim_user in similar_users:
        liked_books = interaction_matrix.loc[sim_user][interaction_matrix.loc[sim_user] > 3].index.tolist()
        recommended_books.extend(liked_books)

    # Remove books the target user has already rated
    already_rated = interaction_matrix.loc[user_id][interaction_matrix.loc[user_id] > 0].index.tolist()
    final_books = [book for book in recommended_books if book not in already_rated]

    # If no books are left after filtering, return empty list
    if not final_books:
        return []

    # Step 4.5: Recommend top N books (most frequently liked)
    return pd.Series(final_books).value_counts().head(top_n).index.tolist()

# Test for a few users
for user in df['user_id'].unique()[:5]:
    print(f"User-User Recommendations for User {user}: {user_user_knn_recommendations(user)}")


User-User Recommendations for User A3HHXRELK8BHQG: ['B002HJV4PM', 'B002YX0PL0']
User-User Recommendations for User A2RGNZ0TRF578I: []
User-User Recommendations for User A3S0H2HV6U1I7F: []
User-User Recommendations for User AC4OQW3GZ919J: []
User-User Recommendations for User A3C9V987IQHOQD: []


### 3. Implement Item-Item Collaborative Filtering

In [7]:
# Compute item-item cosine similarity
item_similarity = pd.DataFrame(cosine_similarity(interaction_matrix.T),
                               index=interaction_matrix.columns,
                               columns=interaction_matrix.columns)


In [8]:
def item_item_recommendations(user_id, top_n=5, sim_n=5):
    
    # Get books rated > 3 by the user
    liked_books = interaction_matrix.loc[user_id][interaction_matrix.loc[user_id] > 3].index
    recommendations = []

    
    for book in liked_books:
        similar_books = item_similarity[book].sort_values(ascending=False)
        top_similar_books = similar_books.iloc[1:sim_n+1].index 
        for similar_book in top_similar_books:
            if similar_book not in liked_books:
                recommendations.append(similar_book)

    # Return top recommended books based on frequency
    return pd.Series(recommendations).value_counts().head(top_n).index.tolist()

# Test on 5 users
for user in interaction_matrix.index[:5]:
    print(f"User {user} => Item-Item Recommendations: {item_item_recommendations(user)}")

User A H Kobayashi => Item-Item Recommendations: []
User A K P => Item-Item Recommendations: ['553508784', '110199715X', '545931908', '1250787653', '1635575583']
User A Reviewer => Item-Item Recommendations: ['553508784', '545931908', '1250787653', '1450805752', '1635575583']
User A Slater => Item-Item Recommendations: ['006332752X', 'B003VYBP9M', 'B003VTZTUI', 'B003VWCCTQ', 'B003VYAY6C']
User A0089401235VSN3Z6F3HK => Item-Item Recommendations: ['B0036Z9YEE', 'B00427ZJ1C', 'B003OIBGSU', 'B002RKRMSY', 'B002RKSZJO']


### 4. Matrix Factorization

In [9]:
#Apply SVD to reduce dimensions (20 features)
svd = TruncatedSVD(n_components=20, random_state=42)
compressed_matrix = svd.fit_transform(interaction_matrix)

#Rebuild the matrix (predict missing ratings)
reconstructed_matrix = np.dot(compressed_matrix, svd.components_)
predicted_ratings_df = pd.DataFrame(reconstructed_matrix, 
                                    index=interaction_matrix.index, 
                                    columns=interaction_matrix.columns)


In [10]:
#Recommend books for a user based on predicted ratings
def svd_recommendations(user_id, top_n=5):
    # Books the user has already rated
    rated_books = interaction_matrix.loc[user_id][interaction_matrix.loc[user_id] > 0].index
    
    # Get predicted ratings for books the user hasn't rated
    predictions = predicted_ratings_df.loc[user_id].drop(rated_books)
    
    # Return top N recommended book IDs
    return predictions.sort_values(ascending=False).head(top_n).index.tolist()

for user in interaction_matrix.index[:5]:
    recommendations = svd_recommendations(user)
    print(f"User {user} => SVD Recommendations: {recommendations}")


User A H Kobayashi => SVD Recommendations: ['547199457', '078525224X', '694003611', '545261244', '1635575605']
User A K P => SVD Recommendations: ['547199457', '078525224X', '694003611', '545261244', '1635575605']
User A Reviewer => SVD Recommendations: ['547199457', '078525224X', '694003611', '545261244', '1635575605']
User A Slater => SVD Recommendations: ['B000WSFBO0', 'B000R93D4Y', 'B001DOHZ5A', 'B002F3PPVE', 'B002BDT64A']
User A0089401235VSN3Z6F3HK => SVD Recommendations: ['B002RKRMSY', 'B000SN6IOQ', 'B000JQUT8S', 'B002RKSZJO', 'B000JMKXYW']


### 5. Evaluation & Observations

In [12]:
def compare_models(user_id):
    print(f"\nRecommendations for User {user_id}")
    print("Books already rated:", list(interaction_matrix.loc[user_id][interaction_matrix.loc[user_id] > 0].index))
    print("User-User CF Recommendations:", user_user_knn_recommendations(user_id))
    print("Item-Item CF Recommendations:", item_item_recommendations(user_id))
    print("SVD Recommendations:", svd_recommendations(user_id))
    print()

# Evaluate a few users
for uid in interaction_matrix.index[:3]:
    compare_models(uid)



Recommendations for User A H Kobayashi
Books already rated: ['1250787653']
User-User CF Recommendations: []
Item-Item CF Recommendations: []
SVD Recommendations: ['547199457', '078525224X', '694003611', '545261244', '1635575605']


Recommendations for User A K P
Books already rated: ['1450805752']
User-User CF Recommendations: []
Item-Item CF Recommendations: ['553508784', '110199715X', '545931908', '1250787653', '1635575583']
SVD Recommendations: ['547199457', '078525224X', '694003611', '545261244', '1635575605']


Recommendations for User A Reviewer
Books already rated: ['110199715X']
User-User CF Recommendations: []
Item-Item CF Recommendations: ['553508784', '545931908', '1250787653', '1450805752', '1635575583']
SVD Recommendations: ['547199457', '078525224X', '694003611', '545261244', '1635575605']

